In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import time
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K
from tensorflow.keras.utils import to_categorical
import ipywidgets as widgets
from ipywidgets import interact

# **Exercise 1: conditional variational autoencoder (CVAE)**
Define and train a conditional variational autoencoder to generate handwritten digits given the digit label as input type:
1. define a CVAE model implementing the **CVAE** class;
2. execute the training process;
3. generate different handwritten digits using the CVAE decoder.

## **Digit labels one hot encoding**
To avoid the model to misinterpret the digit labels, labels are conveniently converted into one hot encoding representation using the [**to_categorical**](https://keras.io/api/utils/python_utils/#tocategorical-function) function provided by Keras.

In [ ]:
train_y_one_hot = to_categorical(train_y,category_count)
val_y_one_hot=to_categorical(val_y,category_count)
test_y_one_hot=to_categorical(test_y,category_count)

print('Train label one hot encoding shape: ',train_y_one_hot.shape)
print('Validation label one hot encoding shape: ',val_y_one_hot.shape)
print('Test label one hot encoding shape: ',test_y_one_hot.shape)

## **Model definition**
The following image shows the architecture of a generic CVAE.

<img src=https://biolab.csr.unibo.it/ferrara/Courses/DL/Tutorials/GenerativeModels/cvae_architecture.png width="500">

Implement the following class to create a CVAE model given:
- the number of input features (*input_count*);
- the dimension of input type (*condition_count*);
- the number of neurons for each hidden layer (*neuron_count_per_hidden_layer*);
- the dimension of the latent space (*encoded_dim*);
- the string identifier of the activation function of the hidden layers (*hidden_activation*);
- the string identifier of the activation function of the output layer (*output_activation*);
- the weight of the regularization term in the VAE loss (*kl_coefficient*).

Both encoder and decoder need to receive two inputs. To this purpose, the Keras [**Concatenate**](https://keras.io/api/layers/merging_layers/concatenate/) layer can be used to concatenate multiple inputs into a single tensor.

In [ ]:
class CVAE(keras.Model):
    def __init__(self, input_count, condition_count, neuron_count_per_hidden_layer, encoded_dim, hidden_activation, output_activation, kl_coefficient, **kwargs):
        super(CVAE, self).__init__(**kwargs)

        #KL coefficient hyperparameter
        self.kl_coefficientg = kl_coefficient
        
        # Encoder
        encoder_input = keras.Input(shape=(input_count, ), name='encoder_input')
        encoder_cond_input = keras.Input(shape=(condition_count, ), name='encoder_cond_input')

        conc_encoder_input = keras.layers.Concatenate()[encoder_input, encoder_cond_input]

        prev_layer = conc_encoder_input

        for neuron_count in neuron_count_per_hidden_layer:
            hidden_layer = layers.Dense(neuron_count, activation = hidden_activation)(prev_layer)
            prev_layer = hidden_layer
        
        mu = layers.Dense(encoded_dim, name='mu')(prev_layer)
        log_var = layers.Dense(encoded_dim, name='log_var')(prev_layer)

        self.encoder = keras.Model([encoder_input, encoder_cond_input], [mu, log_var], name='encoder')

        # Decoder
        decoder_input_sample = keras.Input(shape=(encoded_dim, ), name='decoder_input')
        decoder_input = keras.Concatenate()[decoder_input_sample, encoder_cond_input]


        prev_layer=decoder_input
        for neuron_count in reversed(neuron_count_per_hidden_layer):
          hidden_layer=layers.Dense(neuron_count,activation=hidden_activation)(prev_layer)
          prev_layer=hidden_layer

        decoder_output_layer=layers.Dense(input_count,activation=output_activation, name='decoder_output')(prev_layer)

        self.decoder = keras.Model(decoder_input, decoder_output_layer, name='decoder')

        self.sampling_layer = layers.Lambda(self.sampling, output_shape=(encoded_dim), name='s')


    def sampling(self,args):
        mu, log_var = args
        batch_size = K.shape(mu)[0]
        dim = K.int_shape(mu)[1]
        epsilon = K.random_normal(shape=(batch_size, dim), mean=0., stddev=1.0)
        return K.exp(0.5 * log_var) * epsilon + mu

    def call(self, inputs):
        input_data, input_condition = inputs
        encoder_output = self.encoder(inputs)

        mu = encoder_output[0]
        log_var = encoder_output[1]
        s = self.sampling_layer(encoder_output)
        vae_output = self.decoder(s)

        # Calculate VAE Loss components
        reconstruction_loss = K.sum(K.square(input_data - vae_output), axis=-1)
        kl_loss = 0.5 * K.sum(K.square(mu) + K.exp(log_var) - log_var - 1, axis = -1)
        total_loss = reconstruction_loss + self.kl_coefficient * kl_loss

        self.add_loss(total_loss)

        return vae_out# latent space of dim 2 useful for visualization, but it's a little bit too small



In [5]:
# tle bit too small
cvae = CVAE(train_x_flatten.shape[1], [256,128], 2, 'sigmoid','sigmoid',1)

NameError: name 'train_x_flatten' is not defined